In [ ]:
import os
import gc
import cv2
import time
import random
from monai.inferers import sliding_window_inference

# For data manipulation
import numpy as np
import pandas as pd

# Pytorch Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
from torch.cuda import amp

# Utils
from tqdm.auto import tqdm

# For Image Models
import timm
from timm.utils import ModelEmaV2

# Albumentations for augmentations
import albumentations as A
from albumentations.pytorch import ToTensorV2

import warnings
warnings.filterwarnings("ignore")

## using gpu:1
os.environ['CUDA_VISIBLE_DEVICES'] = '0'

def seed_everything(seed=123):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
seed_everything()

In [ ]:
class Customize_Model(nn.Module):
    def __init__(self, model_name, cls):
        super().__init__()
        self.cls= cls
        self.model = timm.create_model(model_name, 
                                       pretrained=True, 
                                       num_classes=0, 
                                       drop_rate= CFG['drop_out'], 
                                       drop_path_rate= CFG['drop_path']
                                    )
        self.linear = nn.Linear(self.model.num_features, cls)
        
    def forward(self, image):
        x = self.model(image)
        x = self.linear(x)
        return x
    

from transformers import Dinov2ForImageClassification, Dinov2Config
class huggingface_model(nn.Module):
    def __init__(self, cls):
        super().__init__()
        print("Use HuggingFace DINOv2 classification model")
        self.cls = cls
        self.model = Dinov2ForImageClassification.from_pretrained(
            "facebook/dinov2-small-imagenet1k-1-layer"
        )

        in_dim = self.model.classifier.in_features
        self.model.classifier = nn.Linear(in_dim, cls)

    def forward(self, images):
        out = self.model(images).logits
        return out
    
    
class Slide_Window_Model(nn.Module):
    def __init__(self, model_name, cls):
        super().__init__()
        self.cls= cls
        self.model = timm.create_model(model_name, 
                                       pretrained=True, 
                                       num_classes=cls, 
                                       drop_rate= CFG['drop_out'], 
                                       drop_path_rate= CFG['drop_path'])
        
    def forward(self, image):
        x = self.model(image)
        return x if self.training else x.view(-1, self.cls, 1, 1)
    
# x = torch.rand(2,3,512,512)
# model = huggingface_model(cls=3)
# out = model(x)
# print(out.shape)  # torch.Size([2, 3])

In [ ]:
def get_train_transform(img_size):
    return A.Compose([
        A.SmallestMaxSize(max_size=img_size, interpolation=3, p=1),
        # A.Resize(img_size, img_size),
#         A.RandomCrop(CFG['img_crop'], CFG['img_crop'], p=1),
        
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
        A.CLAHE(clip_limit=4.0, p=0.7),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=10, p=0.5),
        A.OneOf([
            A.MotionBlur(blur_limit=5),
            A.MedianBlur(blur_limit=5),
            A.GaussianBlur(blur_limit=5),
            A.GaussNoise(var_limit=(5.0, 30.0)),
        ], p=0.7),
        
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        # A.Transpose(p=0.5),
        A.OneOf([
            A.GridDistortion(num_steps=5, distort_limit=0.05, p=1.0),
            A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=1.0)
        ], p=0.3),
        A.ShiftScaleRotate(shift_limit=0.15, scale_limit=0.05, rotate_limit= 15,
                                        interpolation=cv2.INTER_LINEAR, border_mode=0, p=0.7),
        ToTensorV2(p=1.0),
    ])


def get_test_transform(img_size):
    return A.Compose([
        A.SmallestMaxSize(max_size=img_size, interpolation=3, p=1),
        # A.Resize(img_size, img_size),
        ToTensorV2(p=1.0),
    ])

In [ ]:
class Customize_Dataset(Dataset):
    def __init__(self, df, transforms=None, training=False):
        self.df = df
        self.transforms = transforms
        self.training= training
    
    def mixup_aug(self, img_1, mask_1, 
                        img_2, mask_2):
        """
        img: numpy array of shape (height, width,channel)
        mask: numpy array of shape (height, width,channel)
        """
        ## mixup
        weight= np.random.beta(a=0.5, b=0.5)
        img= img_1*weight + img_2*(1-weight)
        mask= mask_1*weight + mask_2*(1-weight)
        return img.astype(np.uint8), mask
    
    def read_data(self, data):
        img = cv2.imread(data['image_path'].replace('..','.'))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        label= [0]*CFG['num_classes']
        cls = data['label']
        label[cls]= 1
        label= np.array(label)
        return img, label
    
    def __getitem__(self, index):
        data = self.df.loc[index]
        img, label= self.read_data(data)
        
        # use mixup
        if self.training and np.random.rand() >= (1-CFG['mixup']):
            img_1= img
            label_1= np.array(label)
            while True:
                indx= np.random.randint(len(self.df))
                data= self.df.loc[indx]
                img_2, label_2= self.read_data(data)
                if label_1.argmax(0)!=label_2.argmax(0): break
            img, label= self.mixup_aug(img_1, label_1, 
                                       img_2, label_2)
        
        if self.transforms:
            img = self.transforms(image=img)["image"]
            
        return {
            'image': torch.tensor(img/255, dtype=torch.float32),
            'label': torch.tensor(label, dtype=torch.float32),
        }
    
    def __len__(self):
        return len(self.df)

In [ ]:
class Customize_loss(nn.Module):
    def  __init__(self):
        super().__init__()
        self.CrossEntropy= nn.CrossEntropyLoss(weight= None, label_smoothing=0.)
        self.MAE = nn.L1Loss()
        self.MSE = nn.MSELoss()
    
    def forward(self, y_pred, y_true):
        loss= self.CrossEntropy(y_pred, y_true)
        return loss

In [ ]:
def train_epoch(dataloader, model, criterion, optimizer, model_ema):
    scaler= amp.GradScaler()
    model.train()

    ep_loss= []
    for i, data in enumerate(tqdm(dataloader)):

        imgs= data['image'].to('cuda')
        labels= data['label'].to('cuda')
        
        with amp.autocast():
            preds= model(imgs)
            loss= criterion(preds, labels)
            ep_loss.append(loss.item())
            loss/= CFG['gradient_accumulation']
            scaler.scale(loss).backward()
            
            if (i+1) % CFG['gradient_accumulation']== 0:
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
            if model_ema: model_ema.update(model)
                
    return np.mean(ep_loss)

In [ ]:
from sklearn.metrics import recall_score, roc_auc_score
def Accuracy(all_pred, all_label):
    all_pred= all_pred.argmax(1)
    acc= list((all_label==all_pred)+0).count(1) / len(all_label)
    return acc

def Mean_Recall(all_pred, all_label):
    all_pred= all_pred.argmax(1)
    recall= recall_score(all_label, all_pred, average='macro')
    return recall

def AUC(all_pred, all_label):
    auc= roc_auc_score(all_label, all_pred, multi_class='ovo')
    return auc


def valid_epoch(dataloader, model, criterion):
    model.eval()
    
    ep_loss= []
    all_pred= []
    all_label= []
    for i, data in enumerate(tqdm(dataloader)):

        imgs= data['image'].to('cuda')
        labels= data['label'].to('cuda')
        all_label.extend(labels.cpu().numpy())
        
        with torch.no_grad():
            preds= model(imgs)
            loss= criterion(preds, labels)
            ep_loss.append(loss.item())
        all_pred.extend(preds.cpu().softmax(dim=-1).numpy())
        
    
    ## caculate metrics
    all_label= np.array(all_label).argmax(1)
    all_pred= np.array(all_pred)
    print(all_label.shape, all_pred.shape)
    acc= Accuracy(all_pred, all_label)
    print(f'accuracy: {acc}')
    recall= Mean_Recall(all_pred, all_label)
    print(f'mean_recall: {recall}')
    auc= AUC(all_pred[:,1], all_label)
    print(f'AUC: {auc}')
    
    score= auc
    return np.mean(ep_loss), score

# CFG

In [ ]:
CFG= {
    'fold': 0,
    'epoch': 150,
    'model_name': 'tf_efficientnet_b0.ns_jft_in1k',
    
    'img_size': 224,
    'img_crop': None,
    
    'batch_size': 16,
    'gradient_accumulation': 1,
    'gradient_checkpoint': False,
    'drop_out': 0.3,
    'drop_path': 0.2,
    'mixup': 0.,
    'EMA': 0.995,
    
    'lr': 3e-4,
    'weight_decay': 0.,
    
    'num_classes': 2,
    'load_model':  False, #'./train_model/cv0_best.pth',
    'save_model': './train_model'
}

# Prepare Dataset

In [ ]:
df= pd.read_csv('./Data/train_fold.csv')
train_df= df[df['fold']!=CFG['fold']].reset_index(drop=True)
valid_df= df[df['fold']==CFG['fold']].reset_index(drop=True)

train_df['label'] = 1
valid_df['label'] = 0
train_df1 = train_df.sample(frac=0.5, random_state=42).reset_index(drop=True)
valid_df1 = train_df.drop(train_df1.index) 
train_df2 = valid_df.sample(frac=0.5, random_state=42).reset_index(drop=True)
valid_df2 = valid_df.drop(train_df2.index) 
train_df = pd.concat([train_df1, train_df2]).reset_index(drop=True)
valid_df = pd.concat([valid_df1, valid_df2]).reset_index(drop=True)

print(f'train dataset: {len(train_df)}')
print(f'valid dataset: {len(valid_df)}')

train_dataset= Customize_Dataset(train_df, get_train_transform(CFG['img_size']), training=True)
valid_dataset= Customize_Dataset(valid_df, get_test_transform(CFG['img_size']), training=False)

train_loader= DataLoader(train_dataset, batch_size= CFG['batch_size'], shuffle=True, num_workers=0)
valid_loader= DataLoader(valid_dataset, batch_size=1, shuffle=False, num_workers=0)
train_df.head()

In [ ]:
import matplotlib.pyplot as plt

data= train_dataset[0]
img= data['image']
label= data['label']
print(label)
plt.imshow(img.permute(1,2,0).numpy())
plt.show()

# Train

In [ ]:
## create model
if CFG['load_model']:
    print(f"load_model: {CFG['load_model']}")
    model= torch.load(CFG['load_model'], map_location= 'cuda')
else:
    model= Customize_Model(CFG['model_name'], CFG['num_classes'])
    # model = huggingface_model(CFG['num_classes'])
    
if CFG['gradient_checkpoint']: 
    print('use gradient checkpoint')
    model.model.set_grad_checkpointing(enable=True)
    
## EMA
model.to('cuda')
if CFG['EMA']:
    print(f"Use EMA: {CFG['EMA']}")
    model_ema= ModelEmaV2(model, decay=CFG['EMA'])
    model_ema.to('cuda')
else:
    model_ema= type('model_ema', (object,), {'module':{}})
    
## hyperparameter
criterion= Customize_loss()
optimizer= optim.AdamW(model.parameters(), lr= CFG['lr'], weight_decay= CFG['weight_decay'])

## start training
best_score= 0
train_losses= []
val_losses= []
val_score= []
for ep in range(1, CFG['epoch']+1):
    print(f'\nep: {ep}')
    
    if CFG['EMA']: train_loss= train_epoch(train_loader, model, criterion, optimizer, model_ema)
    else: 
        train_loss= train_epoch(train_loader, model, criterion, optimizer, False)
        model_ema.module= model
    valid_loss, valid_mae= valid_epoch(valid_loader, model_ema.module, criterion)
    print(f'train loss: {round(train_loss, 5)}')
    print(f'valid loss: {round(valid_loss, 5)}, valid_r2: {round(valid_mae, 5)}')
    
    train_losses.append(train_loss)
    val_losses.append(valid_loss)
    val_score.append(valid_mae)
    
    if valid_mae >= best_score:
        best_score= valid_mae
        # torch.save(model_ema.module, f"{CFG['save_model']}/cv{CFG['fold']}_best.pth")
        print(f'model save at score: {round(best_score, 5)}')
        
    ## save model every epoch
    # torch.save(model_ema.module, f"{CFG['save_model']}/cv{CFG['fold']}_ep{ep}.pth")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(5,3))
plt.title('train/valid loss')
plt.plot(train_losses)
plt.plot(val_losses)
plt.show()

plt.figure(figsize=(5,3))
plt.title('val score')
plt.ylim(0, 1)
plt.plot(val_score)
plt.show()